# 6.3 怎麼訓練實用的 tokenizer？

# 第 6 章：自己的中英文 tokenizer

前置：字元ID與下一字loss。Tokenizer訓練是統計切詞規則，不是訓練LLM。早期與CLI使用可還原ByteTokenizer；BPE在本章獨立學習，換模型時要同步詞表與checkpoint。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：常見卡片合併：不是先懂詞義**

每次合併一對常見相鄰符號。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/bpe.svg")))

**先想一想：**訓練沒見過的emoji會變成UNK嗎？

Byte-level BPE先提供所有256種byte，再學常見組合；因此未見中文與emoji也能以較細的token表示。不要加會改原文的normalizer。

**把直覺寫成數學：**encode/decode契約是原文UTF-8 bytes還原，不保證每token各自是完整字。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders

tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=280, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
tok.train_from_iterator(["貓看狗。", "a cat sees a dog."], trainer)
text = "小鳥🙂new"
print(tok.encode(text).tokens)
assert tok.decode(tok.encode(text).ids) == text

**如何讀結果：**詞表不足256時不能達成此設定；這是極小示範，不是推薦自然語料配方。

**只改一件事：**只換成一個完全未見文字，驗證還原。
